In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score


In [ ]:
# ── 0.1 Imports ─────────────────────────────────────────────────────────────
import os, sys, json, time, warnings, logging
import random
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (
    StratifiedKFold, train_test_split, cross_val_score
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
import joblib

# XGBoost
from xgboost import XGBClassifier

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback
)
from transformers.trainer_utils import get_last_checkpoint

# Embeddings
import gensim.downloader as api

# NLP
import nltk
import re, unicodedata
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download(["stopwords", "wordnet", "punkt"], quiet=True)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
print("Imports OK")

In [ ]:
# just to set seed
def set_seed(seed: int = 42):
    """Fix all random seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG["seed"])
print("Seed set.")

In [ ]:
# remove links , mentions and new line characters, emojis
# # Clean hashtags at the end of the sentence, and keep those in the middle of the sentence by removing just the # symbol
import re
import string
import emoji
import contractions
from langdetect import detect, LangDetectException

# Rimuove emoji
def strip_emoji(text):
    return emoji.replace_emoji(text, replace='')

# Rimuove punteggiatura, links, mentions, newline, ma mantiene stopwords
def strip_all_entities(text):
    text = re.sub(r'\r|\n', ' ', text.lower())  # lowercase + newline
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)  # mentions e link
    text = re.sub(r'[^\x00-\x7f]', '', text)  # non-ASCII
    banned_list = string.punctuation
    table = str.maketrans('', '', banned_list)
    text = text.translate(table)
    return text

# Pulizia hashtag
def clean_hashtags(tweet):
    # Rimuove hashtag alla fine
    new_tweet = re.sub(r'(\s+#[[\w-]+)+s*$', '', tweet).strip()
    # Rimuove solo # all'interno della frase
    new_tweet = re.sub(r'#([\w-]+)', r'\1', new_tweet).strip()
    return new_tweet

# Filtra caratteri speciali come & e $
def filter_chars(text):
    return ' '.join('' if ('$' in word) or ('&' in word) else word for word in text.split())

# Rimuove spazi multipli
def remove_mult_spaces(text):
    return re.sub(r"\s\s+", " ", text)

# Tiene solo testi in inglese
def filter_non_english(text):
    try:
        lang = detect(text)
    except LangDetectException:
        lang = "unknown"
    return text if lang == "en" else ""

# Espande contrazioni
def expand_contractions(text):
    return contractions.fix(text)

# Rimuove numeri
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

# Rimuove parole troppo corte
def remove_short_words(text, min_len=2):
    words = text.split()
    long_words = [word for word in words if len(word) >= min_len]
    return ' '.join(long_words)

# Sostituisce parole allungate
def replace_elongated_words(text):
    regex_pattern = r'\b(\w+)((\w)\3{2,})(\w*)\b'
    return re.sub(regex_pattern, r'\1\3\4', text)

# Rimuove punteggiatura ripetuta
def remove_repeated_punctuation(text):
    return re.sub(r'[\?\.\!]+(?=[\?\.\!])', '', text)

# Rimuove spazi extra
def remove_extra_whitespace(text):
    return ' '.join(text.split())

# Rimuove short URLs comuni
def remove_url_shorteners(text):
    return re.sub(r'(?:http[s]?://)?(?:www\.)?(?:bit\.ly|goo\.gl|t\.co|tinyurl\.com|tr\.im|is\.gd|cli\.gs|u\.nu|url\.ie|tiny\.cc|alturl\.com|ow\.ly|bit\.do|adoro\.to)\S+', '', text)

# Strip spazi iniziali/finali
def remove_spaces_tweets(tweet):
    return tweet.strip()

# Rimuove tweet troppo corti (<3 parole)
def remove_short_tweets(tweet, min_words=3):
    words = tweet.split()
    return tweet if len(words) >= min_words else ""

# Funzione principale di cleaning
def clean_tweet(tweet):
    tweet = strip_emoji(tweet)
    tweet = expand_contractions(tweet)
    tweet = filter_non_english(tweet)
    tweet = strip_all_entities(tweet)
    tweet = clean_hashtags(tweet)
    tweet = filter_chars(tweet)
    tweet = remove_mult_spaces(tweet)
    tweet = remove_numbers(tweet)
    tweet = remove_short_words(tweet)
    tweet = replace_elongated_words(tweet)
    tweet = remove_repeated_punctuation(tweet)
    tweet = remove_extra_whitespace(tweet)
    tweet = remove_url_shorteners(tweet)
    tweet = remove_spaces_tweets(tweet)
    tweet = remove_short_tweets(tweet)
    tweet = ' '.join(tweet.split())  # rimuove eventuali spazi multipli residui
    return tweet

In [ ]:
# ── 2.1 Preprocessing Functions ──────────────────────────────────────────────
STOP = set(stopwords.words("english"))  # [TASK-SPECIFIC] change language if needed
lemmatizer = WordNetLemmatizer()

def normalize_text(text: str) -> str:
    """
    Minimal cleaning — used as input for Transformers.
    Keeps punctuation and casing information the subword tokenizer relies on.
    """
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"http\S+|www\S+", " ", text)          # URLs
    text = re.sub(r"@\w+|#\w+", " ", text)                # mentions / hashtags
    text = re.sub(r"[^a-zA-Z0-9\s!?.,\'\']", " ", text)   # special chars
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess(text: str,
               remove_stopwords: bool = True,
               lemmatize: bool = True) -> str:
    """
    Full cleaning — used for classical ML and embedding-based models.
    Applies stopword removal and lemmatization on top of normalize_text.
    """
    text   = normalize_text(text)
    tokens = text.split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP]
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)


# Smoke test
sample = "This is a GREAT movie!! http://example.com @user #review"
print("Raw         :", sample)
print("Normalized  :", normalize_text(sample))
print("Preprocessed:", preprocess(sample))

In [ ]:
# ── 4.1 Results Registry ─────────────────────────────────────────────────────
RESULTS = {}  # {"ModelName | split": {metric: value}}


# ── 4.2 Classification Evaluation ────────────────────────────────────────────
def evaluate_clf(model_name: str, y_true, y_pred,
                 y_prob=None, label_encoder=None, split: str = "test") -> dict:
    """
    Computes and prints classification metrics; stores them in RESULTS.

    Parameters
    ----------
    model_name    : display name for the model
    y_true        : ground-truth encoded labels
    y_pred        : predicted encoded labels
    y_prob        : class probabilities shape (n, 2) — binary tasks only, for ROC-AUC
    label_encoder : fitted sklearn LabelEncoder (for human-readable class names)
    split         : "val" or "test"

    Returns
    -------
    dict of computed metrics
    """
    acc       = accuracy_score(y_true, y_pred)
    f1_macro  = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_weight = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    auc = (
        roc_auc_score(y_true, y_prob[:, 1])
        if (y_prob is not None and y_prob.shape[1] == 2) else None
    )

    print(f"\n{'='*55}")
    print(f"  {model_name} | {split.upper()}")
    print(f"{'='*55}")
    print(f"  Accuracy      : {acc:.4f}")
    print(f"  F1 (macro)    : {f1_macro:.4f}")
    print(f"  F1 (weighted) : {f1_weight:.4f}")
    if auc is not None:
        print(f"  ROC-AUC       : {auc:.4f}")

    class_names = label_encoder.classes_.tolist() if label_encoder else None
    print()
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    metrics = {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weight}
    if auc is not None:
        metrics["roc_auc"] = auc

    RESULTS[f"{model_name} | {split}"] = metrics
    return metrics


# ── 4.3 Confusion Matrix Plot ─────────────────────────────────────────────────
def plot_confusion_matrix_clf(y_true, y_pred, model_name: str, label_encoder=None):
    """
    Plots raw-count and row-normalised confusion matrices side by side.
    Saves figure to outputs/.
    """
    cm     = confusion_matrix(y_true, y_pred)
    labels = label_encoder.classes_ if label_encoder else np.unique(y_true)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, data, title, fmt in zip(
        axes, [cm, cm_norm], ["Raw Counts", "Normalized"], ["d", ".2f"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title(title)

    plt.suptitle(f"Confusion Matrix — {model_name}", fontsize=13)
    plt.tight_layout()
    fname = CFG["output_dir"] / f"cm_{model_name.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=120)
    plt.show()


# ── 4.4 Training History Plot ────────────────────────────────────────────────
def plot_history(history: dict, model_name: str):
    """
    Plots loss and F1-macro curves over epochs.

    Expected history keys:
        "train_loss"   : list[float]
        "val_loss"     : list[float]
        "val_f1_macro" : list[float]
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["train_loss"], label="Train", marker="o")
    axes[0].plot(history["val_loss"],   label="Val",   marker="o")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["val_f1_macro"], marker="o", color="orange")
    axes[1].set_title("Val F1 Macro")
    axes[1].set_xlabel("Epoch")

    plt.suptitle(model_name, fontsize=13)
    plt.tight_layout()
    fname = CFG["output_dir"] / f"history_{model_name.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=120)
    plt.show()


# ── 4.5 Results Summary Table ────────────────────────────────────────────────
def print_results_table():
    """Prints a formatted comparison table of all models stored in RESULTS."""
    if not RESULTS:
        print("RESULTS is empty — run evaluate_clf first.")
        return

    all_cols = sorted({k for v in RESULTS.values() for k in v.keys()})
    header = f"{'Model':<35}" + "".join(f"{c:>14}" for c in all_cols)
    sep = "=" * len(header)
    print(f"\n{sep}\n{header}\n{sep}")
    for name, metrics in RESULTS.items():
        row = f"{name:<35}" + "".join(
            f"{metrics.get(c, float('nan')):>14.4f}" for c in all_cols
        )
        print(row)
    print(sep)


print("Evaluation utilities ready.")

In [ ]:
# ── 6.1 Vocabulary Builder ────────────────────────────────────────────────────
# Restore missing vocab_size configuration parameter from the original CFG
# This value was overwritten by a previous cell's CFG redefinition.
# Assuming original value from global config (cell -Ht-1nLAqoRl).
CFG["vocab_size"] = 30_000
CFG["pad_token"] = "<PAD>"
CFG["unk_token"] = "<UNK>"

def build_vocab(texts, max_vocab: int = 30_000,
                pad_token: str = "<PAD>", unk_token: str = "<UNK>") -> dict:
    """
    Builds word → index mapping from tokenised training texts.
    Index 0 = PAD (all-zeros row), Index 1 = UNK.
    """
    counter = Counter()
    for text in texts:
        counter.update(str(text).split())

    vocab = {pad_token: 0, unk_token: 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab


vocab = build_vocab(
    X_train,
    max_vocab=CFG["vocab_size"],
    pad_token=CFG["pad_token"],
    unk_token=CFG["unk_token"]
)
print(f"Vocabulary size: {len(vocab):,}")

In [ ]:
# ── 6.2 Embedding Matrix ──────────────────────────────────────────────────────
# Restore missing embedding configuration parameters from the original CFG
# These values were overwritten by a previous cell's CFG redefinition.
# Assuming original values from global config (cell -Ht-1nLAqoRl).
CFG["embedding_model"] = "glove-wiki-gigaword-100" # or "fasttext-wiki-news-subwords-300"
CFG["embedding_dim"] = 100

def load_embedding_matrix(vocab: dict, model_name: str, embed_dim: int) -> np.ndarray:
    """
    Downloads a pre-trained gensim embedding model and builds a numpy matrix
    aligned with `vocab`. Unknown words get random uniform initialisations.

    Parameters
    ----------
    vocab      : {word: index} mapping
    model_name : gensim key (e.g. "glove-wiki-gigaword-100")
    embed_dim  : embedding dimensionality

    Returns
    -------
    np.ndarray of shape (vocab_size, embed_dim)
    """
    print(f"Loading {model_name} ...")
    wv = api.load(model_name)

    matrix = np.zeros((len(vocab), embed_dim), dtype=np.float32)
    hits = misses = 0
    for word, idx in vocab.items():
        if word in wv:
            matrix[idx] = wv[word]
            hits += 1
        else:
            matrix[idx] = np.random.uniform(-0.25, 0.25, embed_dim)
            misses += 1

    matrix[0] = 0.0  # PAD stays all-zeros
    print(f"Coverage: {hits/(hits+misses)*100:.1f}%  (hits={hits}, misses={misses})")
    return matrix


embedding_matrix = load_embedding_matrix(
    vocab, CFG["embedding_model"], CFG["embedding_dim"]
)
print(f"Embedding matrix shape: {embedding_matrix.shape}")

In [ ]:
# ── 6.3 Text → Padded Token-ID Sequences ─────────────────────────────────────
# Restore missing max_seq_len configuration parameter from the original CFG

CFG["max_seq_len"] = 128

def texts_to_sequences(texts, vocab: dict, max_len: int,
                       unk_token: str = "<UNK>") -> np.ndarray:
    """
    Converts a list of strings to a (N, max_len) int32 numpy array.
    Truncates long sequences; pads short ones with 0 (PAD index).
    """
    unk_idx = vocab.get(unk_token, 1)
    seqs = np.zeros((len(texts), max_len), dtype=np.int32)
    for i, text in enumerate(texts):
        tokens = str(text).split()[:max_len]
        for j, tok in enumerate(tokens):
            seqs[i, j] = vocab.get(tok, unk_idx)
    return seqs


seq_train = texts_to_sequences(X_train, vocab, CFG["max_seq_len"])
seq_val   = texts_to_sequences(X_val,   vocab, CFG["max_seq_len"])
seq_test  = texts_to_sequences(X_test,  vocab, CFG["max_seq_len"])

print(f"seq_train: {seq_train.shape} | seq_val: {seq_val.shape} | seq_test: {seq_test.shape}")

In [ ]:
# ── 6.4 PyTorch Dataset & DataLoaders ────────────────────────────────────────
# Restore missing dl_batch_size configuration parameter from the original CFG
# This value was overwritten by a previous cell's CFG redefinition.
# Assuming original value from global config (cell -Ht-1nLAqoRl).
CFG["dl_batch_size"] = 64

class TextSequenceDataset(Dataset):
    """
    Wraps padded token-ID sequences and integer class labels for classification.

    Parameters
    ----------
    sequences : np.ndarray shape (N, max_len)
    labels    : np.ndarray shape (N,) — integer class indices
    """
    def __init__(self, sequences: np.ndarray, labels: np.ndarray):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def make_loaders(seq_tr, y_tr, seq_va, y_va, seq_te, y_te,
                 batch_size: int = 64):
    """Creates train / val / test DataLoaders."""
    loader_train = DataLoader(TextSequenceDataset(seq_tr, y_tr),
                              batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
    loader_val   = DataLoader(TextSequenceDataset(seq_va, y_va),
                              batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    loader_test  = DataLoader(TextSequenceDataset(seq_te, y_te),
                              batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return loader_train, loader_val, loader_test


loader_train, loader_val, loader_test = make_loaders(
    seq_train, y_train, seq_val, y_val, seq_test, y_test,
    batch_size=CFG["dl_batch_size"]
)
print("DataLoaders ready.")

In [ ]:
# ── 7.0 Logging Setup ─────────────────────────────────────────────────────────
def get_logger(model_name: str) -> logging.Logger:
    log_path = CFG["log_dir"] / f"{CFG['task']}_{model_name.replace(' ', '_')}.log"

    logger = logging.getLogger(model_name)
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.propagate = False

    fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    ))

    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(message)s"))

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# ── 7.1 Single Epoch Helpers ──────────────────────────────────────────────────
def _train_epoch(model, loader, optimizer, criterion, device):
    """One training pass. Returns average cross-entropy loss."""
    model.train()
    total_loss = 0.0
    for seqs, labels in loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(seqs), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * seqs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def _eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for seqs, labels in loader:
        seqs, labels = seqs.to(device), labels.to(device)
        logits = model(seqs)
        total_loss += criterion(logits, labels).item() * seqs.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, f1_macro, np.array(all_preds)


# ── 7.2 Main Training Loop ────────────────────────────────────────────────────
def train_model(model, loader_tr, loader_va, model_name: str,
                epochs: int = 20, lr: float = 1e-3,
                patience: int = 3, device: str = "cpu",
                resume: bool = True) -> dict:
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

    logger = get_logger(model_name)
    tb_dir = CFG["log_dir"] / "tensorboard" / f"{CFG['task']}_{model_name.replace(' ', '_')}"
    writer = SummaryWriter(log_dir=tb_dir)

    history    = {"train_loss": [], "val_loss": [], "val_f1_macro": []}
    best_f1    = -np.inf
    no_improve = 0
    start_epoch = 1

    # --- checkpoint paths
    safe_name = model_name.replace(" ", "_")
    ckpt_dir = CFG["model_dir"] / safe_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = ckpt_dir / "best.pt"
    last_ckpt_path = ckpt_dir / "last.pt"

    # --- optional resume
    if resume and last_ckpt_path.exists():
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        history = ckpt.get("history", history)
        best_f1 = ckpt.get("best_f1", best_f1)
        no_improve = ckpt.get("no_improve", 0)
        start_epoch = ckpt.get("epoch", 0) + 1
        logger.info(f"Resumed from {last_ckpt_path} (epoch={start_epoch-1}, best_f1={best_f1:.4f})")

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        tr_loss           = _train_epoch(model, loader_tr, optimizer, criterion, device)
        va_loss, va_f1, _ = _eval_epoch(model, loader_va, criterion, device)
        elapsed           = time.time() - t0

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_f1_macro"].append(va_f1)
        scheduler.step(va_f1)

        writer.add_scalars("loss", {"train": tr_loss, "val": va_loss}, epoch)
        writer.add_scalar("val/f1_macro", va_f1, epoch)
        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)

        # always save LAST checkpoint (for recovery)
        last_payload = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_f1": best_f1,
            "no_improve": no_improve,
            "history": history,
            "model_name": model_name,
            "task": CFG["task"],
        }
        torch.save(last_payload, last_ckpt_path)

        # save BEST checkpoint
        if va_f1 > best_f1:
            best_f1 = va_f1
            no_improve = 0
            best_payload = {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_f1": best_f1,
                "history": history,
                "model_name": model_name,
                "task": CFG["task"],
            }
            torch.save(best_payload, best_ckpt_path)
            tag = " ✓  <-- new best"
        else:
            no_improve += 1
            tag = ""

        logger.info(
            f"Epoch {epoch:>3}/{epochs} | tr_loss={tr_loss:.4f} | "
            f"va_loss={va_loss:.4f} | val_f1={va_f1:.4f} | {elapsed:.1f}s{tag}"
        )

        if no_improve >= patience:
            logger.info(f"Early stopping triggered at epoch {epoch}.")
            break

    # restore BEST weights from disk
    if best_ckpt_path.exists():
        best_ckpt = torch.load(best_ckpt_path, map_location=device)
        model.load_state_dict(best_ckpt["model_state"])
        logger.info(f"Restored best checkpoint | val_f1={best_ckpt['best_f1']:.4f} | path={best_ckpt_path}")

    writer.close()
    logger.info("Training complete.")
    return history


# ── 7.3 Batch Prediction Helper ───────────────────────────────────────────────
@torch.no_grad()
def predict(model, loader, device: str = "cpu") -> np.ndarray:
    model.eval()
    all_preds = []
    for seqs, _ in loader:
        all_preds.extend(model(seqs.to(device)).argmax(dim=1).cpu().numpy())
    return np.array(all_preds)


print("Training loop utilities ready.")

In [ ]:
# ── 8.1 EmbeddingMLP ─────────────────────────────────────────────────────────
class EmbeddingMLP(nn.Module):
    """
    Mean-pools pre-trained token embeddings, then applies a two-layer MLP.
    Serves as the lightweight "GloVe + MLP" step in the pipeline.

    Parameters
    ----------
    embedding_matrix : np.ndarray, shape (vocab_size, embed_dim)
    num_classes      : number of output classes
    hidden_dim       : hidden layer width
    dropout          : dropout probability
    freeze_emb       : if True, embedding weights are not updated during training
    """
    def __init__(self, embedding_matrix: np.ndarray, num_classes: int,
                 hidden_dim: int = 256, dropout: float = 0.3,
                 freeze_emb: bool = False):
        super().__init__()
        _, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=freeze_emb, padding_idx=0
        )
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)       # (B, L, 1) — zero out PAD
        emb  = self.embedding(x) * mask             # (B, L, D)
        doc  = emb.sum(1) / mask.sum(1).clamp(min=1)  # masked mean → (B, D)
        return self.fc(doc)


# ── 8.2 TextCNN ───────────────────────────────────────────────────────────────
class TextCNN(nn.Module):
    """
    Parallel Conv1d filters over token embeddings with global max-pooling.

    Parameters
    ----------
    embedding_matrix : np.ndarray, shape (vocab_size, embed_dim)
    num_classes      : number of output classes
    num_filters      : number of filters per kernel size
    kernel_sizes     : list of filter heights (e.g. [2, 3, 4])
    dropout          : dropout probability
    freeze_emb       : if True, embedding weights are frozen
    """
    def __init__(self, embedding_matrix: np.ndarray, num_classes: int,
                 num_filters: int = 128, kernel_sizes: list = None,
                 dropout: float = 0.3, freeze_emb: bool = False):
        super().__init__()
        if kernel_sizes is None:
            kernel_sizes = [2, 3, 4]

        _, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=freeze_emb, padding_idx=0
        )
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x).permute(0, 2, 1)    # (B, D, L)
        pooled = [
            F.relu(conv(emb)).max(dim=2).values      # (B, F)
            for conv in self.convs
        ]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))


print("EmbeddingMLP and TextCNN architectures defined.")

In [ ]:
# ── 9.1 Additive Attention ────────────────────────────────────────────────────
class AdditiveAttention(nn.Module):
    """
    Computes a weighted average over LSTM hidden states.
    Weights are learned via a single linear layer + softmax.
    Padding positions are masked out before softmax.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, h, mask=None):
        scores = self.attn(h).squeeze(-1)                       # (B, L)
        if mask is not None:
            scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)                  # (B, L)
        context = (weights.unsqueeze(-1) * h).sum(dim=1)        # (B, H)
        return context, weights


# ── 9.2 BiLSTM + Attention ────────────────────────────────────────────────────
class BiLSTMAttention(nn.Module):
    """
    Bidirectional LSTM with additive attention for text classification.

    Parameters
    ----------
    embedding_matrix : np.ndarray, shape (vocab_size, embed_dim)
    num_classes      : number of output classes
    hidden_dim       : LSTM hidden state size per direction
    num_layers       : number of stacked LSTM layers
    dropout          : dropout probability
    freeze_emb       : freeze pre-trained embeddings during training
    """
    def __init__(self, embedding_matrix: np.ndarray, num_classes: int,
                 hidden_dim: int = 128, num_layers: int = 2,
                 dropout: float = 0.3, freeze_emb: bool = False):
        super().__init__()
        _, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=freeze_emb, padding_idx=0
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.attention = AdditiveAttention(hidden_dim * 2)   # *2 for bidirectional
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        mask = (x != 0)                              # (B, L) True = real token
        emb  = self.dropout(self.embedding(x))       # (B, L, D)
        h, _ = self.lstm(emb)                        # (B, L, 2*H)
        ctx, _ = self.attention(h, mask)             # (B, 2*H)
        return self.fc(self.dropout(ctx))


print("BiLSTMAttention architecture defined.")

In [ ]:
def set_seed(seed: int = 42):
    """Fix all random seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ── 8.4 Train TextCNN ────────────────────────────────────────────────────────
# Restore missing configuration parameters from the original CFG
# These values were overwritten by a previous cell's CFG redefinition.
# Assuming original values from global config (cell -Ht-1nLAqoRl).
CFG["cnn_num_filters"] = 128
CFG["cnn_kernel_sizes"] = [2, 3, 4]
CFG["dropout"] = 0.3
CFG["dl_epochs"] = 20
CFG["dl_lr"] = 1e-3
CFG["dl_patience"] = 3
CFG["device"] = (
    "mps"  if torch.backends.mps.is_available() and torch.backends.mps.is_built()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
# Restore Path objects for directories
CFG["output_dir"] = Path("outputs/")
CFG["model_dir"] = Path("models/")
CFG["log_dir"] = Path("logs/")

set_seed(CFG["seed"])
cnn_model = TextCNN(
    embedding_matrix=embedding_matrix,
    num_classes=CFG["num_labels"],
    num_filters=CFG["cnn_num_filters"],
    kernel_sizes=CFG["cnn_kernel_sizes"],
    dropout=CFG["dropout"]
)
print(f"TextCNN params: {sum(p.numel() for p in cnn_model.parameters()):,}")

cnn_history = train_model(
    cnn_model, loader_train, loader_val,
    model_name="GloVe CNN",
    epochs=CFG["dl_epochs"], lr=CFG["dl_lr"],
    patience=CFG["dl_patience"], device=CFG["device"]
)
plot_history(cnn_history, "GloVe CNN")

cnn_preds = predict(cnn_model, loader_test, CFG["device"])
evaluate_clf("GloVe CNN", y_test, cnn_preds, label_encoder=le)
plot_confusion_matrix_clf(y_test, cnn_preds, "GloVe CNN", le)

In [ ]:
# ── 10.1 HuggingFace Dataset ──────────────────────────────────────────────────
class HFTextDataset(Dataset):
    """
    PyTorch Dataset wrapping a HuggingFace tokenizer for classification.
    Compatible with HF Trainer and standard DataLoader.

    Parameters
    ----------
    texts     : list/array of raw strings
    labels    : encoded integer class indices
    tokenizer : HuggingFace tokenizer
    max_len   : maximum token length
    """
    def __init__(self, texts, labels, tokenizer, max_len: int = 256):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding="max_length",
            max_length=max_len, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


# ── 10.2 Compute Metrics for HF Trainer ──────────────────────────────────────
def compute_metrics(eval_pred) -> dict:
    """
    Passed to HuggingFace Trainer. Returns F1-macro and accuracy.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "accuracy": accuracy_score(labels, preds)
    }


print("HF Dataset and compute_metrics ready.")

In [ ]:
# ── 11.3 Bar Chart Comparison ─────────────────────────────────────────────────
def plot_model_comparison(results: dict, metric: str = "f1_macro",
                          split: str = "test"):
    """
    Horizontal bar chart comparing all models on `metric` for the given `split`.

    Parameters
    ----------
    results : RESULTS dict
    metric  : column to plot (e.g. "f1_macro", "accuracy")
    split   : "test" or "val" — filters RESULTS entries
    """
    filtered = {
        k.replace(f" | {split}", ""): v[metric]
        for k, v in results.items()
        if split in k and metric in v
    }
    if not filtered:
        print(f"No results for split='{split}', metric='{metric}'.")
        return

    names  = list(filtered.keys())
    values = list(filtered.values())
    order  = np.argsort(values)

    fig, ax = plt.subplots(figsize=(9, max(4, len(names) * 0.55)))
    bars = ax.barh(
        [names[i] for i in order],
        [values[i] for i in order],
        color=sns.color_palette("viridis", len(names))
    )
    for bar, val in zip(bars, [values[i] for i in order]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)

    ax.set_xlabel(metric.replace("_", " ").title())
    ax.set_title(f"Model Comparison — {metric} ({split.upper()} set)")
    ax.set_xlim(0, min(1.05, max(values) * 1.12))
    plt.tight_layout()
    plt.savefig(CFG["output_dir"] / f"comparison_{metric}_{split}.png", dpi=120)
    plt.show()


plot_model_comparison(RESULTS, metric="f1_macro", split="test")
plot_model_comparison(RESULTS, metric="accuracy", split="test")

In [ ]:
# ── 11.4 Error Analysis — Top Misclassified Samples ──────────────────────────
def error_analysis(df_split: pd.DataFrame, y_true, y_pred,
                   label_encoder=None, n: int = 10) -> pd.DataFrame:
    """
    Returns a DataFrame of the top-n misclassified samples.

    Parameters
    ----------
    df_split      : DataFrame for the relevant split (e.g. df_test)
    y_true, y_pred: encoded labels
    label_encoder : to decode label names
    n             : number of examples to display
    """
    err_mask = y_true != y_pred
    df_err   = df_split[err_mask].copy()
    df_err["y_true"] = y_true[err_mask]
    df_err["y_pred"] = y_pred[err_mask]

    if label_encoder is not None:
        df_err["true_label"] = label_encoder.inverse_transform(df_err["y_true"])
        df_err["pred_label"] = label_encoder.inverse_transform(df_err["y_pred"])
        cols = [CFG["text_col"], "true_label", "pred_label"]
    else:
        cols = [CFG["text_col"], "y_true", "y_pred"]

    return df_err[cols].head(n)


print("=== Error Analysis — BERT Fine-tune ===")
display(error_analysis(df_test, y_test, trf_test_preds, le, n=10))

In [ ]:
# ── 12.1 Save All Artefacts ───────────────────────────────────────────────────
def save_artefacts():
    """Saves CFG, RESULTS, label encoder and vocab to disk."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    # CFG (Path objects → str)
    cfg_serializable = {k: str(v) if isinstance(v, Path) else v
                        for k, v in CFG.items()}
    with open(CFG["output_dir"] / f"cfg_{timestamp}.json", "w") as f:
        json.dump(cfg_serializable, f, indent=2, default=str)

    # RESULTS
    with open(CFG["output_dir"] / f"results_{timestamp}.json", "w") as f:
        json.dump(RESULTS, f, indent=2)

    # Label encoder & vocab
    joblib.dump(le,    CFG["model_dir"] / "label_encoder.pkl")
    joblib.dump(vocab, CFG["model_dir"] / "vocab.pkl")

    print(f"Artefacts saved  →  {CFG['output_dir']}  |  {CFG['model_dir']}")

save_artefacts()

In [ ]:
# TESTING BEST MODEL

MODEL_PATH = str(CFG["model_dir"] / "transformer_best")

# Load once
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

@torch.no_grad()
def predict_one(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    outputs = model(**inputs)
    logits = outputs.logits

    probs = torch.softmax(logits, dim=-1)
    conf, pred_id = torch.max(probs, dim=-1)

    pred_id = int(pred_id.item())
    confidence = float(conf.item())

    # если есть label encoder
    if "le" in globals():
        pred_label = le.inverse_transform([pred_id])[0]
    else:
        pred_label = str(pred_id)

    return pred_label, confidence